**Eksport:** `EXPORT_DESTINATION = "local"` – pobierz bezpośrednio do `data/gee_rasters` (bez Drive/GCS). Alternatywy: `gcs`, `drive`.

# GEE Hex Terrain: Land Cover & Elevation for H3 Hexes

**Dwa etapy (bez zapytań do GEE przy agregacji):**
1. **GEE Export (jednorazowo)** – eksport rastrów do S3 (GEE→GCS→S3)
2. **Lokalne przetwarzanie** – rasterstats, odczyt z S3, zapis do S3

**Dane:** Copernicus 100m land cover, SRTM 30m elevation + slope

**S3:** `gold/gee_hex_terrain/country=ES/snapshot=2019/h3_resolution=6/`

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

# Land cover: Copernicus 100m (years 2015–2019)
LANDCOVER_YEAR = 2019
LANDCOVER_SCALE = 200

# Elevation + slope: SRTM (resample do 100m – szybszy eksport, ~100MB zamiast ~8GB)
ELEVATION_SCALE = 200
INCLUDE_SLOPE = True

# H3 resolutions to process (6=~36km², 7=~5km², 8=~0.7km², 9=~0.1km²)
# Spain: res6 ~14k, res7 ~96k, res8 ~675k, res9 ~5M
H3_RESOLUTIONS = [6, 7, 8, 9]

# Region
COUNTRY = "ES"  # ISO country code for partition
SNAPSHOT = str(LANDCOVER_YEAR)  # partition: land cover year (or date e.g. 2024-02-15)
USE_SPAIN_BOUNDARY = True  # from FAO GAUL level0
FALLBACK_BBOX = [-9.5, 35.9, 4.6, 43.8]  # [min_lon, min_lat, max_lon, max_lat]
FALLBACK_GEOJSON_PATH = None

# Google Earth Engine (required: create project at https://console.cloud.google.com/earth-engine)
GEE_PROJECT = "ie-microsoft-capstone"  # your Cloud project ID

# S3 (same pattern as gbif notebooks)
S3_BUCKET = "ie-datalake"
BRONZE_PREFIX = "bronze/gee_hex_terrain"  # raw GEE output
GOLD_PREFIX = "gold/gee_hex_terrain"  # aggregated stats: country/snapshot/h3_resolution
AWS_PROFILE = "486717354268_PowerUserAccess"
PARQUET_COMPRESSION = "snappy"

# GEE Export (jednorazowo) – rastry na S3
RUN_GEE_EXPORT = True  # False = skip, używaj już zapisanych rastrów na S3

# Gdzie eksportować: "local" (bezpośrednio na dysk) | "gcs" | "drive"
EXPORT_DESTINATION = "local"  # local = pobierz do data/ bez Drive/GCS

GCS_BUCKET = "ie-microsoft-capstone-gee"  # tylko gdy EXPORT_DESTINATION="gcs"
LOCAL_RASTER_DIR = "data/gee_rasters"    # local/drive: katalog na rastry
LOCAL_GRID_N = 6                         # local: siatka NxN kafelków (każdy <32MB)

# Rastry na S3 (s3://S3_BUCKET/S3_RASTER_PREFIX/)
S3_RASTER_PREFIX = "bronze/gee_rasters"
LANDCOVER_TIF = "landcover_spain.tif"
ELEVATION_TIF = "elevation_spain.tif"
SPAIN_GEOJSON = "spain_boundary.geojson"

In [ ]:
%pip install -q earthengine-api h3 pandas pyarrow s3fs rasterio rasterstats geopandas google-cloud-storage requests

In [3]:
from __future__ import annotations

import json
import logging
from pathlib import Path

import ee
import h3
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("gee_hex")

fs = s3fs.S3FileSystem(profile=AWS_PROFILE)
s3_raster_prefix = f"{S3_BUCKET}/{S3_RASTER_PREFIX}"

if RUN_GEE_EXPORT:
    try:
        ee.Initialize(project=GEE_PROJECT)
        log.info("Earth Engine initialized (project=%s)", GEE_PROJECT)
    except Exception as e:
        log.error("Run: earthengine authenticate\n%s", e)
        raise

12:34:36 [INFO] Earth Engine initialized (project=ie-microsoft-capstone)


## 1) Spain geometry

In [4]:
def get_spain_geometry() -> ee.Geometry:
    """Spain boundary from FAO GAUL level0."""
    countries = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0")
    spain = countries.filter(ee.Filter.eq("ADM0_NAME", "Spain")).first()
    return spain.geometry()


def bbox_to_geometry(bbox: list[float]) -> ee.Geometry:
    """[min_lon, min_lat, max_lon, max_lat] -> ee.Geometry.Rectangle."""
    return ee.Geometry.Rectangle(bbox)


def geojson_to_geometry(path: str) -> ee.Geometry:
    """Load GeoJSON file and convert to ee.Geometry."""
    with open(path) as f:
        gj = json.load(f)
    return ee.Geometry(gj)


spain_geojson_s3 = f"s3://{S3_BUCKET}/{S3_RASTER_PREFIX}/{SPAIN_GEOJSON}"

if RUN_GEE_EXPORT:
    if USE_SPAIN_BOUNDARY:
        region = get_spain_geometry()
        log.info("Using Spain boundary from FAO GAUL")
    elif FALLBACK_GEOJSON_PATH:
        region = geojson_to_geometry(FALLBACK_GEOJSON_PATH)
        log.info("Using geometry from %s", FALLBACK_GEOJSON_PATH)
    else:
        region = bbox_to_geometry(FALLBACK_BBOX)
        log.info("Using fallback bbox %s", FALLBACK_BBOX)
    # Save boundary to S3 for local processing (no GEE needed later)
    gj = region.getInfo()
    with fs.open(spain_geojson_s3, "w") as f:
        json.dump(gj, f, indent=2)
    log.info("Saved boundary to %s", spain_geojson_s3)
else:
    with fs.open(spain_geojson_s3, "r") as f:
        region_geojson = json.load(f)
    region = region_geojson  # dla lokalnego H3; GEE nie jest potrzebne
    log.info("Loaded boundary from %s", spain_geojson_s3)

12:34:39 [INFO] Using Spain boundary from FAO GAUL
12:34:40 [INFO] Found credentials in shared credentials file: ~/.aws/credentials
12:34:40 [INFO] Saved boundary to s3://ie-datalake/bronze/gee_rasters/spain_boundary.geojson


## 2) H3 hex grid

In [6]:
def _extract_polygon_rings(gj: dict) -> list:
    """Extract exterior rings from GeoJSON geometry (Polygon, MultiPolygon, GeometryCollection)."""
    geotype = gj.get("type", "")
    if geotype == "Polygon":
        coords = gj.get("coordinates", [])
        return [coords[0]] if coords else []
    if geotype == "MultiPolygon":
        coords = gj.get("coordinates", [])
        return [p[0] for p in coords] if coords else []
    if geotype == "GeometryCollection":
        rings = []
        for g in gj.get("geometries", []):
            rings.extend(_extract_polygon_rings(g))
        return rings
    return []


def get_h3_hexes_for_region(region_ee_or_geojson, resolution: int) -> list[str]:
    """
    Get H3 hex IDs covering the region.
    region_ee_or_geojson: ee.Geometry (calls getInfo) or GeoJSON dict.
    Handles Polygon, MultiPolygon, GeometryCollection (e.g. FAO GAUL Spain).
    """
    gj = region_ee_or_geojson.getInfo() if hasattr(region_ee_or_geojson, "getInfo") else region_ee_or_geojson
    if not gj:
        raise ValueError("Empty geometry")

    polygons = _extract_polygon_rings(gj)
    if not polygons:
        raise ValueError(f"No polygons in geometry type: {gj.get('type', '')}")

    hexes = set()
    for ring in polygons:
        if len(ring) < 3:
            continue
        if ring[0] != ring[-1]:
            ring = list(ring) + [ring[0]]
        geojson = {"type": "Polygon", "coordinates": [ring]}
        try:
            cells = h3.geo_to_cells(geojson, resolution)
            hexes.update(cells)
        except Exception as e:
            log.warning("Polyfill failed: %s", e)
    return sorted(hexes)


def hex_to_ee_feature(h3_index: str, resolution: int) -> dict:
    """Convert H3 index to GeoJSON-like feature for ee.FeatureCollection."""
    # h3 v4: cell_to_boundary returns (lat, lng); GeoJSON needs [lng, lat]
    boundary = h3.cell_to_boundary(h3_index)
    boundary = [[p[1], p[0]] for p in boundary]  # (lat,lng) -> [lng,lat]
    if boundary[0] != boundary[-1]:
        boundary = list(boundary) + [boundary[0]]
    return {
        "type": "Feature",
        "geometry": {"type": "Polygon", "coordinates": [boundary]},
        "properties": {"h3_index": h3_index, "h3_resolution": resolution},
    }


# Copernicus discrete_classification class IDs (Level 1 simplified)
LC_CLASS_IDS = [0, 20, 30, 40, 50, 60, 70, 80, 90, 100, 111, 112, 113, 114, 115, 116, 121, 122, 123, 124, 125, 126, 200]
LC_CLASS_NAMES = {
    0: "unknown", 20: "shrubs", 30: "herbaceous", 40: "crops", 50: "urban",
    60: "bare", 70: "snow_ice", 80: "water_permanent", 90: "wetland", 100: "moss",
    111: "forest_evergreen_needle", 112: "forest_evergreen_broad", 113: "forest_deciduous_needle",
    114: "forest_deciduous_broad", 115: "forest_mixed", 116: "forest_other",
    121: "open_forest_needle", 122: "open_forest_broad", 123: "open_forest_deciduous_needle",
    124: "open_forest_deciduous_broad", 125: "open_forest_mixed", 126: "open_forest_other",
    200: "ocean",
}

## 3) GEE data & aggregation

In [7]:
def load_landcover(year: int) -> ee.Image:
    """Copernicus Land Cover 100m, discrete_classification band."""
    return ee.Image(f"COPERNICUS/Landcover/100m/Proba-V-C3/Global/{year}").select("discrete_classification")


def load_elevation() -> ee.Image:
    """SRTM elevation + optional slope. Cast to Float so bands match (SRTM=Int16, slope=Float32)."""
    dem = ee.Image("USGS/SRTMGL1_003").select("elevation").toFloat()
    if INCLUDE_SLOPE:
        slope = ee.Terrain.slope(dem)
        return dem.addBands(slope)
    return dem


def build_hex_fc(hex_ids: list[str], resolution: int) -> ee.FeatureCollection:
    """Build ee.FeatureCollection from H3 hex IDs."""
    features = [hex_to_ee_feature(h, resolution) for h in hex_ids]
    return ee.FeatureCollection(features)


def add_landcover_proportions_simple(fc: ee.FeatureCollection, lc_image: ee.Image) -> ee.FeatureCollection:
    """reduceRegions with frequencyHistogram. Proportions computed in Python after getInfo."""
    return lc_image.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.frequencyHistogram(),
        scale=LANDCOVER_SCALE,
    )


def add_elevation_stats(fc: ee.FeatureCollection, elev_image: ee.Image) -> ee.FeatureCollection:
    """Add elevation_mean and slope_mean per hex."""
    return elev_image.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.mean(),
        scale=ELEVATION_SCALE,
    )


def write_to_s3(df_bronze: pd.DataFrame, df_gold: pd.DataFrame, country: str, snapshot: str, h3_resolution: int) -> None:
    """Write bronze (raw) and gold (aggregated stats) to S3. Partition: country/snapshot/h3_resolution."""
    base_bronze = f"{S3_BUCKET}/{BRONZE_PREFIX}/country={country}/snapshot={snapshot}/h3_resolution={h3_resolution}"
    base_gold = f"{S3_BUCKET}/{GOLD_PREFIX}/country={country}/snapshot={snapshot}/h3_resolution={h3_resolution}"
    table_bronze = pa.Table.from_pandas(df_bronze, preserve_index=False)
    table_gold = pa.Table.from_pandas(df_gold, preserve_index=False)
    pq.write_to_dataset(table_bronze, root_path=f"s3://{base_bronze}", filesystem=fs, existing_data_behavior="delete_matching", compression=PARQUET_COMPRESSION)
    pq.write_to_dataset(table_gold, root_path=f"s3://{base_gold}", filesystem=fs, existing_data_behavior="delete_matching", compression=PARQUET_COMPRESSION)
    log.info("  Bronze: s3://%s", base_bronze)
    log.info("  Gold:   s3://%s (%d rows)", base_gold, len(df_gold))

In [8]:
## 3a) GEE Export – rastry na S3 (GEE→GCS/Drive→S3)

if RUN_GEE_EXPORT:
    import time

    lc_image = load_landcover(LANDCOVER_YEAR)
    elev_image = load_elevation()
    prefix = f"gee_rasters_{COUNTRY}_{SNAPSHOT}"
    drive_folder = "GEE_Exports"

    if EXPORT_DESTINATION == "gcs":
        from google.cloud import storage

        task_lc = ee.batch.Export.image.toCloudStorage(
            image=lc_image.clip(region),
            description=f"landcover_{COUNTRY}_{SNAPSHOT}",
            bucket=GCS_BUCKET,
            fileNamePrefix=f"{prefix}/landcover_spain",
            region=region,
            scale=LANDCOVER_SCALE,
            maxPixels=1e13,
            fileFormat="GeoTIFF",
        )
        task_lc.start()
        log.info("Export land cover: %s", task_lc.id)
        task_el = ee.batch.Export.image.toCloudStorage(
            image=elev_image.clip(region),
            description=f"elevation_{COUNTRY}_{SNAPSHOT}",
            bucket=GCS_BUCKET,
            fileNamePrefix=f"{prefix}/elevation_spain",
            region=region,
            scale=ELEVATION_SCALE,
            maxPixels=1e13,
            fileFormat="GeoTIFF",
        )
        task_el.start()
        log.info("Export elevation+slope: %s", task_el.id)

        for task, name in [(task_lc, "landcover"), (task_el, "elevation")]:
            while task.active():
                log.info("  %s: czekam...", name)
                time.sleep(30)
            status = task.status()
            if status.get("state") == "COMPLETED":
                gcs_blob = f"{prefix}/{name}_spain.tif"
                gcs_client = storage.Client(project=GEE_PROJECT)
                bucket = gcs_client.bucket(GCS_BUCKET)
                blob = bucket.blob(gcs_blob)
                data = blob.download_as_bytes()
                s3_key = f"{S3_RASTER_PREFIX}/{name}_spain.tif"
                with fs.open(f"s3://{S3_BUCKET}/{s3_key}", "wb") as f:
                    f.write(data)
                log.info("  %s → s3://%s/%s", name, S3_BUCKET, s3_key)
            else:
                log.error("  %s failed: %s", name, status.get("error_message", status))

    elif EXPORT_DESTINATION == "local":  # pobierz bezpośrednio na dysk (kafelki)
        import io
        import zipfile
        import requests
        import rasterio
        from rasterio.merge import merge as rasterio_merge
        from shapely.geometry import shape

        out_dir = Path(LOCAL_RASTER_DIR)
        out_dir.mkdir(parents=True, exist_ok=True)
        tiles_dir = out_dir / "tiles"
        tiles_dir.mkdir(exist_ok=True)

        shp = shape(region.getInfo())
        minx, miny, maxx, maxy = shp.bounds
        n = LOCAL_GRID_N
        dx, dy = (maxx - minx) / n, (maxy - miny) / n

        def download_and_merge(name: str, image: ee.Image, scale: int, out_path: Path):
            srcs = []
            for i in range(n):
                for j in range(n):
                    x1, y1 = minx + i * dx, miny + j * dy
                    x2, y2 = minx + (i + 1) * dx, miny + (j + 1) * dy
                    rect = ee.Geometry.Rectangle([x1, y1, x2, y2])
                    clip = image.clip(rect)
                    url = clip.getDownloadURL({"region": rect, "scale": scale, "format": "ZIPPED_GEO_TIFF", "filePerBand": False})
                    tile_path = tiles_dir / f"{name}_{i}_{j}.tif"
                    r = requests.get(url, timeout=120)
                    r.raise_for_status()
                    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                        for fn in z.namelist():
                            if fn.endswith(".tif"):
                                with open(tile_path, "wb") as tf:
                                    tf.write(z.read(fn))
                                break
                    srcs.append(rasterio.open(tile_path))
                    log.info("  %s tile %d/%d", name, i * n + j + 1, n * n)
            merged, out_transform = rasterio_merge(srcs)
            meta = srcs[0].meta.copy()
            for s in srcs:
                s.close()
            meta.update(width=merged.shape[2], height=merged.shape[1], transform=out_transform)
            with rasterio.open(out_path, "w", **meta) as dst:
                dst.write(merged)
            log.info("  %s → %s", name, out_path)

        lc_path = out_dir / LANDCOVER_TIF
        elev_path = out_dir / ELEVATION_TIF
        download_and_merge("landcover", lc_image.clip(region), LANDCOVER_SCALE, lc_path)
        download_and_merge("elevation", elev_image.clip(region), ELEVATION_SCALE, elev_path)

        for name, p in [("landcover", lc_path), ("elevation", elev_path)]:
            s3_key = f"{S3_RASTER_PREFIX}/{p.name}"
            with open(p, "rb") as f:
                data = f.read()
            with fs.open(f"s3://{S3_BUCKET}/{s3_key}", "wb") as f:
                f.write(data)
            log.info("  %s → s3://%s/%s", name, S3_BUCKET, s3_key)

    else:  # drive: GEE→Drive, potem pobierz lokalnie i uruchom komórkę 3b
        task_lc = ee.batch.Export.image.toDrive(
            image=lc_image.clip(region),
            description=f"landcover_{COUNTRY}_{SNAPSHOT}",
            folder=drive_folder,
            region=region,
            scale=LANDCOVER_SCALE,
            maxPixels=1e13,
            fileFormat="GeoTIFF",
            fileNamePrefix="landcover_spain",
        )
        task_lc.start()
        task_el = ee.batch.Export.image.toDrive(
            image=elev_image.clip(region),
            description=f"elevation_{COUNTRY}_{SNAPSHOT}",
            folder=drive_folder,
            region=region,
            scale=ELEVATION_SCALE,
            maxPixels=1e13,
            fileFormat="GeoTIFF",
            fileNamePrefix="elevation_spain",
        )
        task_el.start()
        log.info("Export land cover: %s", task_lc.id)
        log.info("Export elevation+slope: %s", task_el.id)

        for task, name in [(task_lc, "landcover"), (task_el, "elevation")]:
            while task.active():
                log.info("  %s: czekam...", name)
                time.sleep(30)
            status = task.status()
            if status.get("state") == "COMPLETED":
                log.info("  %s: gotowe na Drive (folder: %s)", name, drive_folder)
            else:
                log.error("  %s failed: %s", name, status.get("error_message", status))

        log.info("")
        log.info("Pobierz pliki z drive.google.com (folder '%s'): landcover_spain.tif, elevation_spain.tif", drive_folder)
        log.info("Zapisz je w: %s", LOCAL_RASTER_DIR)
        log.info("Następnie uruchom komórkę 3b (Upload z lokalnego do S3)")

12:34:57 [INFO]   landcover tile 1/36
12:34:58 [INFO]   landcover tile 2/36
12:34:59 [INFO]   landcover tile 3/36


KeyboardInterrupt: 

In [9]:
## 3b) Upload z lokalnego do S3 (gdy używasz Drive)

# Uruchom po pobraniu plików z Drive do LOCAL_RASTER_DIR
if RUN_GEE_EXPORT and EXPORT_DESTINATION == "drive":
    local_dir = Path(LOCAL_RASTER_DIR)
    lc_local = local_dir / LANDCOVER_TIF
    elev_local = local_dir / ELEVATION_TIF
    if lc_local.exists() and elev_local.exists():
        for name, p in [("landcover", lc_local), ("elevation", elev_local)]:
            s3_key = f"{S3_RASTER_PREFIX}/{p.name}"
            with open(p, "rb") as f:
                data = f.read()
            with fs.open(f"s3://{S3_BUCKET}/{s3_key}", "wb") as f:
                f.write(data)
            log.info("%s → s3://%s/%s", name, S3_BUCKET, s3_key)
    else:
        log.warning("Brak plików w %s. Pobierz z Drive: %s, %s", local_dir, LANDCOVER_TIF, ELEVATION_TIF)

## 4) Lokalne przetwarzanie – rasterstats → S3

In [29]:
# ═══════════════════════════════════════════════════════════════════════════════
# PARAMETRY PRZETWARZANIA
# ═══════════════════════════════════════════════════════════════════════════════
# Co przetwarzać: landcover | elevation | oba
PROCESS_LANDCOVER = False
PROCESS_ELEVATION = True
# Wznowienie: (res, hex_start) np. (9, 500000) = od res 9, hexa 500000; None = od początku
RESUME_FROM = None  # (9, 500000)  # odkomentuj po crashu

import tempfile
import geopandas as gpd
import pyarrow.parquet as pq
import rasterio
from rasterstats import zonal_stats
from shapely.geometry import Polygon

need_lc = PROCESS_LANDCOVER
need_elev = PROCESS_ELEVATION
resume = RESUME_FROM  # (res, hex_start) lub None

local_dir = Path(LOCAL_RASTER_DIR)
lc_local = local_dir / LANDCOVER_TIF
elev_local = local_dir / ELEVATION_TIF
lc_s3 = f"{S3_BUCKET}/{S3_RASTER_PREFIX}/{LANDCOVER_TIF}"
elev_s3 = f"{S3_BUCKET}/{S3_RASTER_PREFIX}/{ELEVATION_TIF}"

tmp = None
lc_path = elev_path = None
if need_lc:
    if lc_local.exists():
        lc_path = lc_local
    elif fs.exists(lc_s3):
        tmp = tempfile.mkdtemp() if tmp is None else tmp
        lc_path = Path(tmp) / LANDCOVER_TIF
        fs.get(lc_s3, str(lc_path))
    else:
        raise FileNotFoundError(f"Brak landcover w {local_dir} ani s3://{lc_s3}")
if need_elev:
    if elev_local.exists():
        elev_path = elev_local
    elif fs.exists(elev_s3):
        tmp = tempfile.mkdtemp() if tmp is None else tmp
        elev_path = Path(tmp) / ELEVATION_TIF
        fs.get(elev_s3, str(elev_path))
    else:
        raise FileNotFoundError(f"Brak elevation w {local_dir} ani s3://{elev_s3}")
if not need_lc and not need_elev:
    raise ValueError("Ustaw PROCESS_LANDCOVER=True lub PROCESS_ELEVATION=True")
if lc_path or elev_path:
    log.info("Używam rastrów z %s", local_dir if (lc_path and lc_path.parent == local_dir) else tmp or "S3")
if resume:
    log.info("Wznowienie: res=%d od hexa %d", resume[0], resume[1])

try:
    all_dfs = []
    for res in H3_RESOLUTIONS:
        if resume and res < resume[0]:
            log.info("Resolution %d: pomijam (wznowienie od res %d)", res, resume[0])
            continue
        log.info("Resolution %d: generating hex grid...", res)
        hex_ids = get_h3_hexes_for_region(region, res)
        n_hex = len(hex_ids)
        if resume and res == resume[0]:
            hex_start = min(resume[1], n_hex)
            hex_ids = hex_ids[hex_start:]
            log.info("  %d hexes (wznowienie od indeksu %d, zostaje %d)", n_hex, hex_start, len(hex_ids))
        else:
            log.info("  %d hexes", n_hex)
        if n_hex == 0 or len(hex_ids) == 0:
            continue

        # GeoDataFrame hexów
        geoms = []
        for h in hex_ids:
            boundary = h3.cell_to_boundary(h)
            coords = [[p[1], p[0]] for p in boundary]  # (lat,lng) -> [lng,lat]
            if coords[0] != coords[-1]:
                coords.append(coords[0])
            geoms.append(Polygon(coords))
        gdf = gpd.GeoDataFrame({"h3_index": hex_ids, "h3_resolution": res}, geometry=geoms, crs="EPSG:4326")
        n_chunk = len(hex_ids)

        lc_pct = {f"lc_{cid}_pct": [] for cid in LC_CLASS_IDS}
        elev_means = [None] * n_chunk
        slope_means = [None] * n_chunk if INCLUDE_SLOPE else []

        if need_lc:
            stats_lc = zonal_stats(gdf, str(lc_path), categorical=True, geojson_out=False)
            for s in stats_lc:
                raw = s.get("histogram", {}) or {}
                hist = {int(k): v for k, v in raw.items()}
                total = sum(hist.values()) or 1
                for cid in LC_CLASS_IDS:
                    lc_pct[f"lc_{cid}_pct"].append(hist.get(cid, 0) / total)
        else:
            stats_lc = [{"histogram": {}}] * n_chunk
            for cid in LC_CLASS_IDS:
                lc_pct[f"lc_{cid}_pct"] = [0.0] * n_chunk  # brak landcover – wypełnij zerami

        if need_elev:
            with rasterio.open(elev_path) as src:
                stats_el = zonal_stats(gdf, src.read(1), affine=src.transform, stats="mean", nodata=src.nodata)
                stats_sl = zonal_stats(gdf, src.read(2), affine=src.transform, stats="mean", nodata=src.nodata)
            elev_means = [s.get("mean") for s in stats_el]
            slope_means = [s.get("mean") for s in stats_sl] if INCLUDE_SLOPE else [None] * n_chunk

        # Gold
        gold_rows = [{"h3_index": h, "h3_resolution": res, "elevation_mean": e, "slope_mean": s} for h, e, s in zip(hex_ids, elev_means, slope_means)]
        for k, v in lc_pct.items():
            for i, row in enumerate(gold_rows):
                row[k] = v[i]
        df_gold = pd.DataFrame(gold_rows)

        # Bronze
        bronze_rows = []
        for i, h in enumerate(hex_ids):
            hist = {int(k): v for k, v in (stats_lc[i].get("histogram") or {}).items()}
            bronze_rows.append({
                "h3_index": h, "h3_resolution": res,
                "elevation": elev_means[i], "slope": slope_means[i] if INCLUDE_SLOPE else None,
                "discrete_classification": json.dumps(hist),
            })
        df_bronze = pd.DataFrame(bronze_rows)

        # Przy wznowieniu: dołącz do istniejącej partycji na S3
        if resume and res == resume[0] and resume[1] > 0:
            base_gold = f"{S3_BUCKET}/{GOLD_PREFIX}/country={COUNTRY}/snapshot={SNAPSHOT}/h3_resolution={res}"
            base_bronze = f"{S3_BUCKET}/{BRONZE_PREFIX}/country={COUNTRY}/snapshot={SNAPSHOT}/h3_resolution={res}"
            try:
                existing_gold = pq.read_table(f"s3://{base_gold}", filesystem=fs)
                existing_bronze = pq.read_table(f"s3://{base_bronze}", filesystem=fs)
                df_ex_g = existing_gold.to_pandas()
                df_ex_b = existing_bronze.to_pandas()
                df_ex_g = df_ex_g.iloc[: resume[1]]
                df_ex_b = df_ex_b.iloc[: resume[1]]
                df_gold = pd.concat([df_ex_g, df_gold], ignore_index=True)
                df_bronze = pd.concat([df_ex_b, df_bronze], ignore_index=True)
                log.info("  Scalono z istniejącymi %d wierszami", resume[1])
            except Exception as e:
                log.warning("  Brak istniejącej partycji lub błąd odczytu: %s. Zapisuję tylko nowe.", e)

        all_dfs.append(df_gold)
        write_to_s3(df_bronze, df_gold, COUNTRY, SNAPSHOT, res)
finally:
    if tmp:
        import shutil
        shutil.rmtree(tmp, ignore_errors=True)

00:16:59 [INFO] Używam rastrów z S3
00:16:59 [INFO] Resolution 6: generating hex grid...
00:17:00 [INFO]   13641 hexes
00:17:11 [INFO]   Bronze: s3://ie-datalake/bronze/gee_hex_terrain/country=ES/snapshot=2019/h3_resolution=6
00:17:11 [INFO]   Gold:   s3://ie-datalake/gold/gee_hex_terrain/country=ES/snapshot=2019/h3_resolution=6 (13641 rows)
00:17:11 [INFO] Resolution 7: generating hex grid...
00:17:14 [INFO]   95481 hexes
00:18:30 [INFO]   Bronze: s3://ie-datalake/bronze/gee_hex_terrain/country=ES/snapshot=2019/h3_resolution=7
00:18:30 [INFO]   Gold:   s3://ie-datalake/gold/gee_hex_terrain/country=ES/snapshot=2019/h3_resolution=7 (95481 rows)
00:18:30 [INFO] Resolution 8: generating hex grid...
00:18:41 [INFO]   668392 hexes
00:24:17 [INFO]   Bronze: s3://ie-datalake/bronze/gee_hex_terrain/country=ES/snapshot=2019/h3_resolution=8
00:24:17 [INFO]   Gold:   s3://ie-datalake/gold/gee_hex_terrain/country=ES/snapshot=2019/h3_resolution=8 (668392 rows)
00:24:17 [INFO] Resolution 9: generati

KeyboardInterrupt: 

In [ ]:
if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    log.info("Total: %d rows across %d resolutions", len(df_all), len(all_dfs))
    display(df_all.head(10))

In [ ]:
# (Komórkę powyżej możesz usunąć – duplikat wyświetlania)

## Walidacja Gold Layer

Preview + sprawdzenie czy elevation i land cover zapisały się poprawnie.

In [1]:
# Walidacja gold layer – uruchom po zapisie do S3
import pyarrow.dataset as ds

VALIDATE_RES = 7  # resolution do walidacji (6/7/8/9)
gold_path = f"s3://{S3_BUCKET}/{GOLD_PREFIX}/country={COUNTRY}/snapshot={SNAPSHOT}/h3_resolution={VALIDATE_RES}"

try:
    tbl = ds.dataset(gold_path, filesystem=fs, format="parquet").scanner().to_table()
    df = tbl.to_pandas()
except Exception as e:
    print(f"❌ Nie można załadować gold: {e}")
    raise

print(f"✓ Załadowano {len(df):,} wierszy (res {VALIDATE_RES})")
print(f"  Kolumny: {list(df.columns)}")

# 1) Elevation + slope
elev_ok = "elevation_mean" in df.columns
slope_ok = "slope_mean" in df.columns
print(f"\n📐 Elevation: {'✓' if elev_ok else '✗'} elevation_mean")
print(f"📐 Slope:     {'✓' if slope_ok else '✗'} slope_mean")

if elev_ok:
    n_null = df["elevation_mean"].isna().sum()
    rng = (df["elevation_mean"].min(), df["elevation_mean"].max())
    print(f"   zakres: {rng[0]:.0f}–{rng[1]:.0f} m, null: {n_null}")
if slope_ok:
    n_null = df["slope_mean"].isna().sum()
    rng = (df["slope_mean"].min(), df["slope_mean"].max())
    print(f"   zakres: {rng[0]:.1f}–{rng[1]:.1f}°, null: {n_null}")

# 2) Land cover
lc_cols = [c for c in df.columns if c.startswith("lc_") and c.endswith("_pct")]
print(f"\n🗺️ Land cover: {'✓' if lc_cols else '✗'} {len(lc_cols)} klas (lc_*_pct)")

if lc_cols:
    lc_sum = df[lc_cols].sum(axis=1)
    ok_sum = (lc_sum > 0.99) & (lc_sum < 1.01)
    n_bad = (~ok_sum).sum()
    print(f"   suma % na wiersz: min={lc_sum.min():.3f}, max={lc_sum.max():.3f}")
    if n_bad > 0:
        print(f"   ⚠️ {n_bad} wierszy ma sumę ≠ 1.0")
    else:
        print(f"   ✓ wszystkie wiersze: suma ≈ 1.0")

# 3) Preview
print("\n📋 Preview (5 losowych wierszy):")
sample = df.sample(n=min(5, len(df)), random_state=42)
preview_cols = ["h3_index"]
if "elevation_mean" in df.columns:
    preview_cols.append("elevation_mean")
if "slope_mean" in df.columns:
    preview_cols.append("slope_mean")
preview_cols += (lc_cols[:5] if lc_cols else [])
display(sample[[c for c in preview_cols if c in sample.columns]])

NameError: name 'S3_BUCKET' is not defined